# Embedding model evaluation

Compares candidate embedding models on **retrieval quality**, using what we already have:
- `0. cleaned_data/chunks.parquet` (739 chunks) as the corpus to search over
- `eval/eval_draft.jsonl` (40 questions with known-correct `answer_law`/`answer_article_numbers`) as ground truth

For each model: embed every chunk, embed every question, retrieve top-k chunks by cosine similarity, and check whether a chunk from the *correct* article shows up. Reported as Recall@k and MRR, broken out by same-language vs. cross-lingual questions (the 8 cross-lingual ones are the real differentiator - a weak multilingual model does fine on the 32 same-language questions and falls over on those 8).

**Kernel: `rag311`** (registered Jupyter kernel, Python 3.11, `C:\Users\Mohannad Alsaleh\.conda\envs\rag311`) - has GPU-enabled torch (cu128, detects the RTX 5070), `transformers`, `cohere`, `openai`, and `python-dotenv` installed.

**Self-hosted candidates:**
- `Qwen/Qwen3-Embedding-0.6B`, `BAAI/bge-m3`, `intfloat/multilingual-e5-large`

**Hosted API candidates** (run only if the matching API key is available - otherwise skipped with a message, not an error):
- Cohere `embed-v4.0` - needs `COHERE_API_KEY`. Flagged in earlier research as notably better on Arabic specifically.
- OpenAI `text-embedding-3-small` - needs `OPENAI_API_KEY`. Cheap general-purpose baseline, not Arabic-specialized.

**API keys live in `PROJECT/.env`, never in this notebook.** `.env` is listed in `.gitignore` so it can never be accidentally committed. To set them:
1. Open `.env` in the project root (it already exists, with two empty placeholder lines).
2. Fill in the values: `COHERE_API_KEY=sk-...` and `OPENAI_API_KEY=sk-...` (no quotes, no spaces around `=`).
3. Save the file, then restart the kernel (or just re-run the first cell - `load_dotenv()` re-reads the file every time it's called, unlike a shell env var it doesn't need a fresh process).

**Resumable:** a model already present in `embedding_eval_summary.csv` is skipped on the next run, not recomputed - encoding 739 chunks is the expensive part, and there's no reason to redo it just to pick up one newly-added API key. Add a model's name to `FORCE_RECOMPUTE` in the last cell to force it to redo anyway (e.g. after changing its pooling/prefix logic). This also means the "unauthenticated requests to the HF Hub" warning only shows up for models actually being (re)computed, not ones being skipped.

**Implementation note (self-hosted only):** encoding uses plain `transformers` + `torch`, not `sentence-transformers`. On the machine this was developed on, `sentence-transformers` unconditionally imports `scikit-learn` at package-load time, and Windows 11's **Smart App Control** (enforced there) blocks scikit-learn's compiled `_argkmin_classmode.pyd` for not meeting its Enterprise signing-level requirement - confirmed via the CodeIntegrity event log, not a guess. Rather than ask to weaken a real security control (Smart App Control can't be re-enabled once turned off without reinstalling Windows), pooling is implemented by hand per model:
- **e5**: mean-pooling over token embeddings (attention-mask-weighted), right-padded, 512-token cap (XLM-RoBERTa-large's hard limit - a 1024 cap here previously crashed the CUDA kernel with an out-of-bounds position-embedding index).
- **bge-m3**: `[CLS]`-token pooling, right-padded (matches its official dense-vector convention).
- **Qwen3-Embedding**: last-token pooling, **left**-padded (required so the last real token is always at a fixed position across a batch - this is a causal/decoder-style embedding model, not a BERT-style encoder).

Each model/provider also has its own query/passage formatting convention, made explicit rather than assumed uniform - getting this wrong silently tanks retrieval quality:
- **e5** needs literal `"query: "` / `"passage: "` prefixes.
- **bge-m3** needs no prefix on either side.
- **Qwen3-Embedding** wraps queries in an instruction (`"Instruct: ...\nQuery: ..."`); passages are encoded plain.
- **Cohere** uses an explicit `input_type` parameter (`search_query` vs. `search_document`) instead of text prefixes.
- **OpenAI** `text-embedding-3` models are symmetric - no query/passage distinction.

Results are saved incrementally (one write after each model), keyed by model name, so a crash or a missing API key on a later candidate can't lose earlier results.

Caveat: 40 questions is a small eval set. Treat results as directional - good enough to spot a clear winner/loser, not to split hairs between close scores.

In [ ]:
import json
import gc
import os
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from dotenv import load_dotenv

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
CHUNKS_PATH = PROJECT / "0. cleaned_data" / "chunks.parquet"
EVAL_PATH = PROJECT / "eval" / "eval_draft.jsonl"
RESULTS_DIR = PROJECT / "eval"

# Loads COHERE_API_KEY / OPENAI_API_KEY from PROJECT/.env into os.environ.
# .env is git-ignored - never put real keys directly in this notebook or any
# other tracked file, since this project is headed for a public repo.
load_dotenv(PROJECT / ".env")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

chunks = pd.read_parquet(CHUNKS_PATH)
chunks["article_number"] = chunks["article_number"].astype(str)

eval_questions = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

print("chunks:", chunks.shape, "| eval questions:", eval_questions.shape)

## Reconciling naming between the two files

`chunks.parquet` uses short law codes (`ssl`, `labor96`, `prof19`) from the chunking step; `eval_draft.jsonl` uses the full names used when the eval set was drafted (`Social_Security_Law`, `Work_Law_1996`, `Professional_Work_Law_2019`). A question is scored as "hit" if a retrieved chunk matches on **law + language + article_number**, not article_number alone (article numbers repeat across the 3 laws).

In [ ]:
LAW_MAP = {
    "Work_Law_1996": "labor96",
    "Social_Security_Law": "ssl",
    "Professional_Work_Law_2019": "prof19",
}
assert set(eval_questions["answer_law"].unique()) == set(LAW_MAP.keys())
assert set(chunks["law"].unique()) == set(LAW_MAP.values())

eval_questions["answer_law_code"] = eval_questions["answer_law"].map(LAW_MAP)
eval_questions["is_cross_lingual"] = eval_questions["question_language"] != eval_questions["answer_language"]
print(eval_questions["is_cross_lingual"].value_counts())

## Self-hosted candidates: formatting + pooling

In [ ]:
QWEN_TASK = "Given a question about Jordanian labor law or social security law, retrieve the law article that answers it"

def e5_query(q):     return f"query: {q}"
def e5_passage(p):   return f"passage: {p}"
def bge_query(q):    return q
def bge_passage(p):  return p
def qwen_query(q):   return f"Instruct: {QWEN_TASK}\nQuery: {q}"
def qwen_passage(p): return p

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return summed / counts

def cls_pool(last_hidden, attention_mask):
    return last_hidden[:, 0]

def last_token_pool(last_hidden, attention_mask):
    # requires left-padding so the last real token is always at position -1
    return last_hidden[:, -1]

LOCAL_CANDIDATES = {
    "qwen3-embedding-0.6b": {
        "kind": "local", "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "query_fmt": qwen_query, "passage_fmt": qwen_passage,
        "pool": last_token_pool, "padding_side": "left", "max_length": 1024,
    },
    "bge-m3": {
        "kind": "local", "model_id": "BAAI/bge-m3",
        "query_fmt": bge_query, "passage_fmt": bge_passage,
        "pool": cls_pool, "padding_side": "right", "max_length": 1024,
    },
    "multilingual-e5-large": {
        "kind": "local", "model_id": "intfloat/multilingual-e5-large",
        "query_fmt": e5_query, "passage_fmt": e5_passage,
        "pool": mean_pool, "padding_side": "right", "max_length": 512,  # XLM-RoBERTa-large hard limit
    },
}

## Hosted API candidates

Skipped (not an error) if the matching key isn't set.

In [ ]:
HOSTED_CANDIDATES = {}

if os.environ.get("COHERE_API_KEY"):
    HOSTED_CANDIDATES["cohere-embed-v4"] = {
        "kind": "cohere", "model_id": "embed-v4.0",
    }
else:
    print("COHERE_API_KEY not set - skipping Cohere embed-v4")

if os.environ.get("OPENAI_API_KEY"):
    HOSTED_CANDIDATES["openai-text-embedding-3-small"] = {
        "kind": "openai", "model_id": "text-embedding-3-small",
    }
else:
    print("OPENAI_API_KEY not set - skipping OpenAI text-embedding-3-small")

CANDIDATES = {**LOCAL_CANDIDATES, **HOSTED_CANDIDATES}
print("will evaluate:", list(CANDIDATES.keys()))

## Encoding

One `encode_*` function per `kind`. All return an (n_texts, dim) tensor, L2-normalized, so cosine similarity is just a dot product later regardless of which provider produced it.

In [ ]:
BATCH_SIZE_LOCAL = 16
BATCH_SIZE_API = 90
MAX_RETRIES = 5

def encode_local(model, tokenizer, texts, pool_fn, padding_side, max_length):
    tokenizer.padding_side = padding_side
    embs = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH_SIZE_LOCAL):
            batch = texts[i:i + BATCH_SIZE_LOCAL]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            pooled = pool_fn(out.last_hidden_state, enc["attention_mask"])
            pooled = F.normalize(pooled, p=2, dim=1)
            embs.append(pooled.cpu())
    return torch.cat(embs, dim=0)


def encode_cohere(texts, model_id, input_type):
    import cohere
    client = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
    embs = []
    for i in range(0, len(texts), BATCH_SIZE_API):
        batch = texts[i:i + BATCH_SIZE_API]
        for attempt in range(MAX_RETRIES):
            try:
                resp = client.embed(texts=batch, model=model_id, input_type=input_type, embedding_types=["float"])
                embs.extend(resp.embeddings.float_)
                break
            except Exception as e:
                print(f"  cohere retry {attempt + 1}: {e}")
                time.sleep(2 ** attempt)
        else:
            raise RuntimeError("Cohere embed failed after retries")
    return F.normalize(torch.tensor(embs, dtype=torch.float32), p=2, dim=1)


def encode_openai(texts, model_id):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    embs = []
    for i in range(0, len(texts), BATCH_SIZE_API):
        batch = texts[i:i + BATCH_SIZE_API]
        for attempt in range(MAX_RETRIES):
            try:
                resp = client.embeddings.create(input=batch, model=model_id)
                embs.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                print(f"  openai retry {attempt + 1}: {e}")
                time.sleep(2 ** attempt)
        else:
            raise RuntimeError("OpenAI embed failed after retries")
    return F.normalize(torch.tensor(embs, dtype=torch.float32), p=2, dim=1)

## Evaluation

Retrieve top-k per question, record the rank of the first chunk matching the question's ground-truth `(law, language, article_number)`, and compute Recall@1/5/10 and MRR - overall and split by same-language vs. cross-lingual.

In [ ]:
TOP_K = 10

def evaluate_model(name, cfg):
    print(f"\n=== {name} ({cfg['model_id']}) ===")
    passage_texts = chunks["text"].tolist()
    query_texts = eval_questions["question_text"].tolist()

    if cfg["kind"] == "local":
        tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"])
        model = AutoModel.from_pretrained(cfg["model_id"]).to(DEVICE)
        model.eval()
        passages = [cfg["passage_fmt"](t) for t in passage_texts]
        print(f"  encoding {len(passages)} chunks...")
        corpus_emb = encode_local(model, tokenizer, passages, cfg["pool"], cfg["padding_side"], cfg["max_length"])
        queries = [cfg["query_fmt"](t) for t in query_texts]
        print(f"  encoding {len(queries)} questions...")
        query_emb = encode_local(model, tokenizer, queries, cfg["pool"], cfg["padding_side"], cfg["max_length"])
        del model, tokenizer
    elif cfg["kind"] == "cohere":
        print(f"  encoding {len(passage_texts)} chunks...")
        corpus_emb = encode_cohere(passage_texts, cfg["model_id"], "search_document")
        print(f"  encoding {len(query_texts)} questions...")
        query_emb = encode_cohere(query_texts, cfg["model_id"], "search_query")
    elif cfg["kind"] == "openai":
        print(f"  encoding {len(passage_texts)} chunks...")
        corpus_emb = encode_openai(passage_texts, cfg["model_id"])
        print(f"  encoding {len(query_texts)} questions...")
        query_emb = encode_openai(query_texts, cfg["model_id"])
    else:
        raise ValueError(f"unknown kind: {cfg['kind']}")

    sims = query_emb @ corpus_emb.T  # (n_questions, n_chunks)
    topk = torch.topk(sims, k=TOP_K, dim=1)

    rows = []
    for qi, row in enumerate(eval_questions.itertuples(index=False)):
        correct_articles = set(row.answer_article_numbers)
        rank = None  # 1-indexed rank of first correct hit, None if not in top_k
        for pos, idx in enumerate(topk.indices[qi].tolist(), start=1):
            c = chunks.iloc[idx]
            if c["law"] == row.answer_law_code and c["language"] == row.answer_language and c["article_number"] in correct_articles:
                rank = pos
                break
        rows.append({
            "question_id": row.question_id,
            "is_cross_lingual": row.is_cross_lingual,
            "difficulty": row.difficulty,
            "rank": rank,
        })

    del corpus_emb, query_emb, sims, topk
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def summarize(details, model_name):
    def metrics(df):
        n = len(df)
        if n == 0:
            return {"n": 0, "recall@1": None, "recall@5": None, "recall@10": None, "mrr": None}
        r = df["rank"]
        return {
            "n": n,
            "recall@1":  (r == 1).sum() / n,
            "recall@5":  (r.notna() & (r <= 5)).sum() / n,
            "recall@10": r.notna().sum() / n,
            "mrr":       (1 / r.dropna()).sum() / n,
        }

    out = []
    for label, subset in [
        ("overall", details),
        ("same-language", details[~details["is_cross_lingual"]]),
        ("cross-lingual", details[details["is_cross_lingual"]]),
    ]:
        out.append({"model": model_name, "subset": label, **metrics(subset)})
    return pd.DataFrame(out)

In [ ]:
DETAILS_PATH = RESULTS_DIR / "embedding_eval_details.csv"
SUMMARY_PATH = RESULTS_DIR / "embedding_eval_summary.csv"

# Models already present in the summary CSV are skipped, not recomputed -
# encoding 739 chunks is the expensive part, no reason to redo it on every
# re-run just to pick up one new hosted candidate. Add a model name here to
# force it to redo anyway (e.g. after changing its pooling/prefix logic).
FORCE_RECOMPUTE = set()

already_done = set()
if SUMMARY_PATH.exists():
    already_done = set(pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")["model"].unique())

all_details = []
all_summaries = []

for name, cfg in CANDIDATES.items():
    if name in already_done and name not in FORCE_RECOMPUTE:
        print(f"skipping '{name}' - already in {SUMMARY_PATH.name} (add to FORCE_RECOMPUTE to redo)")
        continue

    details = evaluate_model(name, cfg)
    details["model"] = name
    all_details.append(details)
    all_summaries.append(summarize(details, name))

    # save after every model, keyed by model name, so a later crash/missing
    # key can't lose earlier results and re-running only refreshes what ran
    details_df = pd.concat(all_details, ignore_index=True)
    summary_df = pd.concat(all_summaries, ignore_index=True)
    if DETAILS_PATH.exists():
        prior = pd.read_csv(DETAILS_PATH, encoding="utf-8-sig")
        prior = prior[~prior["model"].isin(details_df["model"].unique())]
        details_df = pd.concat([prior, details_df], ignore_index=True)
    if SUMMARY_PATH.exists():
        prior = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
        prior = prior[~prior["model"].isin(summary_df["model"].unique())]
        summary_df = pd.concat([prior, summary_df], ignore_index=True)
    details_df.to_csv(DETAILS_PATH, index=False, encoding="utf-8-sig")
    summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
    print(f"  saved progress through '{name}'")

pd.set_option("display.width", 120)
print()
print(pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig").round(3).to_string(index=False))

## Results

All 5 candidates, run against the same 739-chunk corpus and 40-question eval set (`eval/embedding_eval_summary.csv`):

| model | subset | n | Recall@1 | Recall@5 | Recall@10 | MRR |
|---|---|---|---|---|---|---|
| **cohere-embed-v4** | overall | 40 | **0.650** | 0.825 | 0.925 | **0.735** |
| cohere-embed-v4 | same-language | 32 | **0.812** | 0.938 | **1.000** | **0.862** |
| cohere-embed-v4 | cross-lingual | 8 | 0.000 | 0.375 | 0.625 | 0.226 |
| **bge-m3** | overall | 40 | 0.475 | 0.850 | 0.925 | 0.640 |
| bge-m3 | same-language | 32 | 0.594 | 0.844 | 0.938 | 0.727 |
| bge-m3 | cross-lingual | 8 | 0.000 | **0.875** | **0.875** | **0.294** |
| qwen3-embedding-0.6b | overall | 40 | 0.525 | 0.825 | 0.850 | 0.651 |
| qwen3-embedding-0.6b | same-language | 32 | 0.656 | 0.875 | 0.906 | 0.757 |
| qwen3-embedding-0.6b | cross-lingual | 8 | 0.000 | 0.625 | 0.625 | 0.229 |
| multilingual-e5-large | overall | 40 | 0.550 | 0.725 | 0.750 | 0.625 |
| multilingual-e5-large | same-language | 32 | 0.688 | 0.875 | 0.906 | 0.771 |
| multilingual-e5-large | cross-lingual | 8 | 0.000 | 0.125 | 0.125 | 0.042 |
| openai-text-embedding-3-small | overall | 40 | 0.525 | 0.675 | 0.725 | 0.592 |
| openai-text-embedding-3-small | same-language | 32 | 0.656 | 0.812 | 0.844 | 0.721 |
| openai-text-embedding-3-small | cross-lingual | 8 | 0.000 | 0.125 | 0.250 | 0.075 |

**Reading it:**
- **cohere-embed-v4** dominates same-language retrieval - perfect Recall@10 (1.000), 81% Recall@1, best overall MRR of all 5.
- **bge-m3** is the strongest on the cross-lingual subset (7/8 found by Recall@10, best cross-lingual MRR) but noticeably weaker than Cohere on same-language.
- **openai-text-embedding-3-small** is the weakest overall, particularly cross-lingual (only 2/8) - consistent with it not being Arabic-specialized.
- Every model scored 0.000 Recall@1 on the cross-lingual subset - none ever ranked the correct cross-lingual chunk #1, across 3 architecturally different self-hosted models plus 2 hosted APIs. Reads as a genuine property of the task, not a bug, but matters if retrieval ever used top-1 only instead of top-k.
- Caveat: the cross-lingual subset is only 8 questions - one flipped result swings that column by 12.5 points. Treat as directional, not a precise leaderboard.

## Decision: Cohere embed-v4

Initial instinct was to favor bge-m3 for its stronger cross-lingual numbers, since a bilingual corpus seemed to make cross-lingual retrieval the important differentiator. That reasoning didn't hold up under closer scrutiny of what "cross-lingual matters" actually referred to here - worth recording why, since the correction is as informative as the result.

**The case for weighting cross-lingual retrieval turned out weaker than it first appeared:**
1. It rested partly on SCL Article 85-bis supposedly existing only in Arabic (added by a 2023 amendment "that the English translation predates," per the extraction notebook's own note). Checked directly: **both languages have 85-bis** (`scl_articles_en.xlsx` and `scl_articles_ar.xlsx` both contain it) - the note was stale, written before the Defect 1 fix that stopped 85-bis from silently overwriting Article 85 in the English file. Confirmed no real content gap here.
2. The other supporting case - the Professional Work Law 2019 having an unamended 2019 English text vs. an amended-through-2024 Arabic text - is a **different problem than cross-lingual retrieval solves**. If a user asks about that law in English, the system will *correctly* retrieve the English article; it'll just be *stale content*, because that's genuinely what the English source says. No amount of cross-lingual retrieval skill fixes a document being out of date - that needs an updated translation or metadata flagging, not a different embedding model. (Confirmed further: none of the 4 Professional Work Law eval questions are even in the cross-lingual subset - they're all same-language. Scope of the actual divergence, per the effective-date metadata checked: only Articles 2-4 confirmed revised; Articles 5-13 unchecked, not confirmed either way.)

With both supporting examples removed or reclassified, there's no demonstrated content-gap case left that cross-lingual retrieval specifically addresses in this corpus - just a speculative hedge against gaps not yet found in the ~265 articles that haven't been manually diffed across languages. That's real but weak evidence next to Cohere's clearly stronger same-language numbers (which cover the vast majority of realistic questions).

**Cost check before committing to a paid API** (using the actual corpus - 739 chunks, 477,249 characters):
- One-time corpus embedding: ~135-140K tokens -> **~$0.015-0.02 total**, once (only re-run if the corpus changes).
- Per-query embedding (recurring, unavoidable with any model choice - every live question has to be embedded before retrieval can run): average question ~20-25 tokens -> **~$0.000003/question**. Even a generously-trafficked demo doesn't approach real money.
- Real tradeoff isn't cost, it's operational: Cohere requires a live API call in the request path for every query the deployed app answers (external dependency, needs a funded production key, not the rate-limited trial key). A self-hosted model (bge-m3) has zero marginal cost and no external dependency, at the cost of running inference on your own infrastructure.

**Decision: use Cohere `embed-v4.0`** for both corpus indexing and query-time embedding. Same-language accuracy is the dominant factor for this corpus's realistic usage, the cost is negligible at this project's scale, and the operational dependency is an accepted tradeoff for a project whose explicit goal is public deployment anyway (see `CLAUDE.md` deployment goal - the backend will already need to be a live service making outbound calls for the generation step, so one more outbound API call for embeddings doesn't add a fundamentally new failure mode).